## **ASSEMBLY ALGORITHMS**

First, we import the necessary libraries and setup the notebook. In this case, we will use some basic libraries for data manipulation and visualization.


In [10]:
import sys

assert sys.version_info >= (3, 7)

In [11]:
from packaging import version
import sklearn

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

In [12]:
import pandas as pd
import numpy as np

Import the regressors that we will train later with the data

In [13]:
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

Before training our algoriths, let's charge the dataset. This dataset contains features to predict the `performance index` of a student.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
df = pd.read_csv("/content/drive/MyDrive/Student_Performance.csv")

In [16]:
df

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,Yes,4,2,23.0
9996,7,64,Yes,8,5,58.0
9997,6,83,Yes,8,5,74.0
9998,9,97,Yes,7,0,95.0


In [17]:
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({"Yes": 1, "No":0})

Let's change the names of the features to snake case.

In [18]:
df.rename(columns={
    "Hours Studied" : "hours_studied",
    "Previous Scores" : "previous_scores",
    "Sleep Hours" : "sleep_hours",
    "Extracurricular Activities":"extracurricular_activities",
    "Sample Question Papers Practiced" : "sample_question_papers"
}, inplace=True)

df

,hours_studied,previous_scores,extracurricular_activities,sleep_hours,sample_question_papers,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,1,4,2,23.0
9996,7,64,1,8,5,58.0
9997,6,83,1,8,5,74.0
9998,9,97,1,7,0,95.0


In this case, our dataset contains the following features:

* `Hours Studied`: The number of hours a student studied.
* `Previous Scores`: The scores obtained by the student in previous exams.
* `Extracurricular Activities`: Whether the student participates in extracurricular activities (Yes/No).
* `Sleep Hours`: The average number of hours the student sleeps per night.
* `Sample Question Papers Practiced`: The number of sample question papers the student has practiced.
* `Performance Index`: The target variable we want to predict, which is a score representing the student's overall performance.

The `Extracurricular Activities` feature is categorical, so we will drop it for simplicity, as we are focusing on regression algorithms that work with numerical data.

In [19]:
df

,hours_studied,previous_scores,extracurricular_activities,sleep_hours,sample_question_papers,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,1,4,2,23.0
9996,7,64,1,8,5,58.0
9997,6,83,1,8,5,74.0
9998,9,97,1,7,0,95.0


Now, we will split our dataset into the features (X) and the target variable (y). The features will include `Hours Studied`, `Previous Scores`, `Sleep Hours`, and `Sample Question Papers Practiced`, while the target variable will be `Performance Index`.

In [20]:
X = df.drop("Performance Index", axis=1)
Y = df['Performance Index']

We have the features and the target variable ready, now we will split our dataset into a training set and a test set. The training set will be used to train our regression algorithms, while the test set will be used to evaluate their performance.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, train_size=0.65, random_state=42)

### **VOTING REGRESSOR**

Let's start with the Voting Regressor. This algorithm combines the predictions of multiple regression models to improve the overall performance. We will use a simple ensemble of three regressors: Support Vector Regressor, Decision Tree Regressor, and K-Nearest Neighbors Regressor.

In [22]:
voting_rgs = VotingRegressor(
    estimators=[
        ('knn', KNeighborsRegressor()),
        ('rf', RandomForestRegressor(random_state=42)),
        ('svr', SVR())
    ]
)
voting_rgs.fit(X_train, y_train)

VotingRegressor(estimators=[('knn', KNeighborsRegressor()),
                            ('rf', RandomForestRegressor(random_state=42)),
                            ('svr', SVR())])

We can see what is the precision of each of the regressors individually, and then we will see how the Voting Regressor performs compared to them.

In [23]:
for name, clf in voting_rgs.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

knn = 0.9836262476070496
rf = 0.9858485836925808
svr = 0.9844961895208484


In general, the three regressors have almost the same precision (~98%), but we can use the cross-validation technique to see how they perform on different subsets of the data.

### **K-FOLD VALIDATION**

In [24]:
from sklearn.model_selection import KFold, cross_val_score

k = KFold(n_splits=5, shuffle=True, random_state=42)
cross_val_score(voting_rgs, X_train, y_train, cv=k)

array([0.98546074, 0.98643294, 0.9859958 , 0.9852692 , 0.98642263])

### **BAGGING REGRESSOR**

The Bagging Regressor is an ensemble method that trains multiple instances of the same regression algorithm on different subsets of the training data and then averages their predictions. We will use the Support Vector Regressor as our base estimator for the Bagging Regressor.

In [25]:
from sklearn.ensemble import BaggingRegressor

bag_rgs = BaggingRegressor(estimator=KNeighborsRegressor(), n_estimators=500, max_samples=100, n_jobs=-1, random_state=42, oob_score=True)

In [26]:
bag_rgs.fit(X_train, y_train)

BaggingRegressor(estimator=KNeighborsRegressor(), max_samples=100,
                 n_estimators=500, n_jobs=-1, oob_score=True, random_state=42)

Let's make a OOB (out-of-bag) evaluation of the Bagging Regressor to see how it performs on the training data without using a separate test set.

In [27]:
bag_rgs.oob_score_

0.9569870681465924

The OOB score threw us a precision of ~95%, which is quite good. Now, let's evaluate the Bagging Regressor with cross-validation.

In [28]:
k = KFold(n_splits=5, shuffle=True, random_state=42)
cross_val_score(bag_rgs, X_train, y_train, cv=k)

array([0.95845064, 0.95619308, 0.95411624, 0.95458442, 0.9585943 ])

## **EVALUATION METRICS**

In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [30]:
voting_rgs_pred = voting_rgs.predict(X_test)
bagging_rgs_pred = bag_rgs.predict(X_test)

Evaluate the metrics for the voting regressor

In [31]:
mae_voting_regressor = mean_absolute_error(y_test, voting_rgs_pred)
mse_voting_regressor = mean_squared_error(y_test, voting_rgs_pred)
r2_voting_regressor = r2_score(y_test, voting_rgs_pred)

print(f"Voting Regressor - MAE: {mae_voting_regressor}")
print(f"Voting Regressor - MSE: {mse_voting_regressor}")
print(f"Voting Regressor - R2 Score: {r2_voting_regressor}")

Voting Regressor - MAE: 1.7476127368980325
Voting Regressor - MSE: 4.860184128758345
Voting Regressor - R2 Score: 0.9869337473950102


Secondly, evaluate the metrics for the bagging regressor.

In [32]:
mae_bagging_regressor = mean_absolute_error(y_test, bagging_rgs_pred)
mse_bagging_regressor = mean_squared_error(y_test, bagging_rgs_pred)
r2_bagging_regressor = r2_score(y_test, bagging_rgs_pred)

print(f'Bagging Regressor - MAE: {mae_bagging_regressor}')
print(f'Bagging Regressor - MSE: {mse_bagging_regressor}')
print(f'Bagging Regressor - R2 Score: {r2_bagging_regressor}')

Bagging Regressor - MAE: 3.1154622857142864
Bagging Regressor - MSE: 15.328150877760004
Bagging Regressor - R2 Score: 0.958791377851074


## **EXPORTING THE MODELS**

Finally, we can export our trained models using the `pickle` library, which allows us to save the models to disk and load them later for making predictions on new data.

In this case, we will create a web service using FastAPI to serve our models and make predictions based on user input. For this, we eill export the both models, the Voting Regressor and the Bagging Regressor, so we can compare their predictions in the web service.

In [33]:
import pickle

First, we will export the Voting Regressor model.

In [34]:
with open('voting_regressor_trained-0.1.0.pkl', "wb") as f:
    pickle.dump(voting_rgs, f)

In [35]:
with open('bagging_regressor_trained-0.1.0.pkl', 'wb') as f:
    pickle.dump(bag_rgs, f)